# Dijet pileup-filter systematic uncertainty

Compare reconstructed dijet pseudorapidity distributions from the nominal `dz1p0`, `Gplus`, and optional `Vtx1` pileup-filter productions for MinimumBias, Jet60, Jet80, and Jet100 data. Full CM distributions are normalized independently to unit integral. Forward/Backward ratios are constructed from unnormalized forward and backward yields with standard independent-error propagation.

The symmetric, double-sided pileup uncertainty is `|Gplus / dz1p0 - 1|`. `Vtx1 / dz1p0` is diagnostic only and never contributes to the uncertainty estimate.


## Environment and imports

Start Jupyter from the repository root with `py-env/bin/python -m jupyter notebook`.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from dataclasses import replace
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'CMakeLists.txt').is_file() and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate the jetAnalysis repository; start Jupyter from its root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import DIJET_DELTA_PHI_SELECTION_LABEL, STANDARD_DIJET_ETA_CUT_INDEX
from hist_analysis.python.dijet_closures import DijetClosureCurve, build_dijet_gen_comparisons
from hist_analysis.python.histogram_io import resolve_data_file
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.plotting import draw_overlay
from hist_analysis.python.root_style import COLORS, DEFAULT_PLOT_STYLE, save_canvas
from hist_analysis.python.systematic_fits import (
    calculate_one_sided_systematic, fit_histogram_variations,
    format_fit_summary_lines, smooth_systematic_running_max,
    write_systematic_csv,
)

ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)


## Configuration

`SYSTEMATIC_EXTRACTION` selects fitted or direct bin-by-bin values. `APPLY_SYSTEMATIC_SMOOTHING` applies the same outward running-maximum convention as the JEU/JER notebooks and defaults to enabled. `DRAW_VTX1` controls only the diagnostic Vtx1 curve. The optional `''`/`'B'` choices affect variation/default comparisons; F/B construction is permanently restricted to independent errors.


In [ ]:
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
DATA_DIRECTION = 'combined'
DATA_SELECTION = 'jetId'
TRIGGERS = ('MinimumBias', 'Jet60', 'Jet80', 'Jet100')
PTAVE_BINS = {
    'MinimumBias': [(60, 80), (80, 100), (100, 120), (120, 180)],
    'Jet60': [(80, 100), (100, 120), (120, 180), (180, 250)],
    'Jet80': [(100, 120), (120, 180), (180, 250), (300, 500)],
    'Jet100': [(120, 180), (180, 250), (300, 500)],
}
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
REBIN_ETA = 2
DRAW_VTX1 = False
SYSTEMATIC_EXTRACTION = 'fit'  # 'fit' or 'bin_by_bin'
APPLY_SYSTEMATIC_SMOOTHING = True
FULL_SMOOTHING_ORIGIN = -0.465 + 0.00001
FORWARD_BACKWARD_RATIO_OPTION = ''  # protected: F / B is never binomial
FULL_COMPARISON_RATIO_OPTION = 'B'   # '' or 'B'
FB_COMPARISON_RATIO_OPTION = 'B'     # '' or 'B'; applies after F/B construction
FULL_FIT_FUNCTION = 'pol2'
FB_FIT_FUNCTION = 'pol1'
FIT_INITIAL_VALUES = {
    'full': {'Gplus / dz1p0': (1.0, 0.0, 0.0), 'Vtx1 / dz1p0': (1.0, 0.0, 0.0)},
    'fb': {'Gplus / dz1p0': (1.0, 0.0), 'Vtx1 / dz1p0': (1.0, 0.0)},
}
FIT_OPTIONS = 'RQS0'          # fit range, result, quiet, no draw
FIT_WEIGHT_OPTION = ''       # 'W': weight 1 for each non-empty bin; '': use bin errors
EFFECTIVE_FIT_OPTIONS = FIT_OPTIONS + FIT_WEIGHT_OPTION
FIT_WEIGHT_TAG = 'weights1' if FIT_WEIGHT_OPTION == 'W' else 'weightsStd'
OUTPUT_CONFIGURATION_TAG = (
    f'fullFit_{FULL_FIT_FUNCTION}_fbFit_{FB_FIT_FUNCTION}'
    f'_{FIT_WEIGHT_TAG}_systCombMax'
)
SHOW_FIT_RESULTS = True
DRAW_GRID = True
SAVE_PNG = False
FULL_RATIO_RANGE = (0.85, 1.15)
FB_RANGE = (0.75, 1.30)
FB_DOUBLE_RATIO_RANGE = (0.85, 1.15)
SYSTEMATIC_Y_RANGE = None
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_PILEUP_SYSTEMATICS_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'systematics_pileup',
))

if SYSTEMATIC_EXTRACTION not in ('fit', 'bin_by_bin'):
    raise ValueError("SYSTEMATIC_EXTRACTION must be 'fit' or 'bin_by_bin'")
if FORWARD_BACKWARD_RATIO_OPTION != '':
    raise ValueError('Forward/Backward construction must use standard independent errors')
if FIT_WEIGHT_OPTION not in ('', 'W'):
    raise ValueError("FIT_WEIGHT_OPTION must be empty or 'W'")
for name, option in (('FULL_COMPARISON_RATIO_OPTION', FULL_COMPARISON_RATIO_OPTION),
                     ('FB_COMPARISON_RATIO_OPTION', FB_COMPARISON_RATIO_OPTION)):
    if option not in ('', 'B'):
        raise ValueError(f'{name} must be empty or B')
if not 0 <= ETA_CUT_INDEX < len(ETA_CUTS):
    raise IndexError(f'Invalid ETA_CUT_INDEX: {ETA_CUT_INDEX}')
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
FILTERS = ('dz1p0', 'Gplus') + (('Vtx1',) if DRAW_VTX1 else ())
DATA_FILES = {
    trigger: {vertex_filter: resolve_data_file(
        DATA_DIR, trigger, DATA_DIRECTION, DATA_SELECTION, vertex_filter
    ) for vertex_filter in FILTERS}
    for trigger in TRIGGERS
}
for trigger, files in DATA_FILES.items():
    for vertex_filter, path in files.items():
        if not path.exists():
            raise FileNotFoundError(f'Missing {trigger} {vertex_filter} ROOT file: {path}')

CURVE = DijetClosureCurve(
    'selection', 'hRecoDijetPtEtaCM_{eta_cut_index}',
    'hRecoDijetPtEtaForward_{eta_cut_index}',
    'hRecoDijetPtEtaBackward_{eta_cut_index}',
)
PLOT_STYLE = replace(DEFAULT_PLOT_STYLE, annotation_text_size=0.026,
                     annotation_line_spacing=0.039, legend_text_size=0.028)
SHAPE_STYLE = {'Gplus': 0, 'Vtx1': 1, 'dz1p0': 2}
RATIO_STYLE = {'Gplus / dz1p0': 0, 'Vtx1 / dz1p0': 1}


## Build projections, comparisons, and pileup uncertainties

The three productions use the same stored histogram keys and projection intervals. Unit-integral normalization is applied only to the full CM shapes. Forward and backward projections remain unnormalized until `Forward / Backward` is constructed.


In [ ]:
def build_filter_histograms(trigger, ptave_range):
    eta_shapes, fb_ratios, selected_keys = {}, {}, {}
    for vertex_filter in FILTERS:
        shapes, ratios, keys = build_dijet_gen_comparisons(
            DATA_FILES[trigger][vertex_filter], (CURVE,), eta_cut_index=ETA_CUT_INDEX,
            ptave_range=ptave_range, nominal='selection', rebin_eta=REBIN_ETA,
            normalization='integral', ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
        )
        eta_shapes[vertex_filter] = shapes['selection']
        fb_ratios[vertex_filter] = ratios['selection']
        selected_keys[vertex_filter] = keys['selection']
    return eta_shapes, fb_ratios, selected_keys


def variation_ratios(histograms, *, option, name_prefix):
    return {
        f'{vertex_filter} / dz1p0': ratio_to_nominal(
            histograms[vertex_filter], histograms['dz1p0'],
            name=f'{name_prefix}_{vertex_filter.lower()}_to_dz1p0', option=option,
        )
        for vertex_filter in FILTERS if vertex_filter != 'dz1p0'
    }


def draw_ratio_with_systematic_band(
    ratios, systematic, *, x_range, y_range, fit_functions,
    output, canvas_name, x_title, annotations,
):
    """Overlay filter/default points, fits, and the symmetric uncertainty band."""
    canvas = draw_overlay(
        ratios, title='', x_title=x_title, y_title='Pileup filter / dz1p0',
        x_range=x_range, y_range=y_range, reference_y=1.0,
        annotations=annotations, grid=DRAW_GRID, overlay_functions=fit_functions,
        style_indices=RATIO_STYLE, style=PLOT_STYLE,
        output=None, save_png=False, canvas_name=canvas_name,
    )
    graph = ROOT.TGraphAsymmErrors(systematic.GetNbinsX())
    for bin_index in range(1, systematic.GetNbinsX() + 1):
        point = bin_index - 1
        graph.SetPoint(point, systematic.GetBinCenter(bin_index), 1.0)
        half_width = systematic.GetBinWidth(bin_index) / 2.0
        uncertainty = systematic.GetBinContent(bin_index)
        graph.SetPointError(point, half_width, half_width, uncertainty, uncertainty)
    graph.SetFillColorAlpha(COLORS[3], 0.30)
    graph.SetLineColor(COLORS[3])
    graph.Draw('E2')
    for ratio in ratios.values():
        ratio.Draw('E1 SAME')
    for function in fit_functions.values():
        function.Draw('SAME')
    canvas._overlay_objects[0].AddEntry(graph, 'Pileup syst. uncrt.', 'f')
    canvas.Modified()
    canvas.Update()
    save_canvas(canvas, output, save_png=SAVE_PNG)
    canvas._overlay_objects.append(graph)
    return canvas


pileup_results = {}
eta_x_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
fb_x_range = (0.0, ETA_CUT + 0.1)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



def analyze_trigger(trigger):
    trigger_results = {}
    for ptave_range in PTAVE_BINS[trigger]:
        low, high = ptave_range
        ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
        eta_cut_tag = int(round(10.0 * ETA_CUT))
        sample_tag = trigger.lower()
        common_tag = f'{sample_tag}_pileupSystematics_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        output_name = lambda plot: (
            f'{sample_tag}_pileupSystematics_{plot}'
            f'_{OUTPUT_CONFIGURATION_TAG}'
            f'_etaCM_{eta_cut_tag}_ptave_{ptave_tag}.pdf'
        )
        systematic_output_tag = (
            'systematic_smoothed'
            if APPLY_SYSTEMATIC_SMOOTHING else 'systematic_nonsmoothed'
        )
        eta_shapes, fb_ratios, selected_keys = build_filter_histograms(trigger, ptave_range)
        eta_variations = variation_ratios(
            eta_shapes, option=FULL_COMPARISON_RATIO_OPTION, name_prefix=f'h_{common_tag}_full')
        fb_variations = variation_ratios(
            fb_ratios, option=FB_COMPARISON_RATIO_OPTION, name_prefix=f'h_{common_tag}_fb')

        eta_functions, eta_summaries = fit_histogram_variations(
            eta_variations, formula=FULL_FIT_FUNCTION, fit_range=(-ETA_CUT, ETA_CUT),
            name_prefix=f'f_{common_tag}_full', fit_options=EFFECTIVE_FIT_OPTIONS,
            initial_values={k: FIT_INITIAL_VALUES['full'][k] for k in eta_variations},
        )
        fb_functions, fb_summaries = fit_histogram_variations(
            fb_variations, formula=FB_FIT_FUNCTION, fit_range=(0.0, ETA_CUT),
            name_prefix=f'f_{common_tag}_fb', fit_options=EFFECTIVE_FIT_OPTIONS,
            initial_values={k: FIT_INITIAL_VALUES['fb'][k] for k in fb_variations},
        )
        use_fit = SYSTEMATIC_EXTRACTION == 'fit'
        eta_systematic_raw = calculate_one_sided_systematic(
            eta_variations['Gplus / dz1p0'], name=f'h_{common_tag}_full_relative_systematic',
            variation_function=eta_functions['Gplus / dz1p0'] if use_fit else None,
            evaluation_range=(-ETA_CUT, ETA_CUT),
        )
        fb_systematic_raw = calculate_one_sided_systematic(
            fb_variations['Gplus / dz1p0'], name=f'h_{common_tag}_fb_relative_systematic',
            variation_function=fb_functions['Gplus / dz1p0'] if use_fit else None,
            evaluation_range=(0.0, ETA_CUT),
        )
        eta_systematic = (smooth_systematic_running_max(
            eta_systematic_raw, name=f'{eta_systematic_raw.GetName()}_smoothed',
            evaluation_range=(-ETA_CUT, ETA_CUT), smoothing_origin=FULL_SMOOTHING_ORIGIN)
            if APPLY_SYSTEMATIC_SMOOTHING else eta_systematic_raw)
        fb_systematic = (smooth_systematic_running_max(
            fb_systematic_raw, name=f'{fb_systematic_raw.GetName()}_smoothed',
            evaluation_range=(0.0, ETA_CUT))
            if APPLY_SYSTEMATIC_SMOOTHING else fb_systematic_raw)
        eta_percent = eta_systematic.Clone(f'{eta_systematic.GetName()}_percent')
        fb_percent = fb_systematic.Clone(f'{fb_systematic.GetName()}_percent')
        eta_percent.SetDirectory(0); fb_percent.SetDirectory(0)
        eta_percent.Scale(100.0); fb_percent.Scale(100.0)
        eta_systematic_csv = write_systematic_csv(
            eta_systematic,
            OUTPUT_DIR / output_name(f'{systematic_output_tag}_full_relative').replace('.pdf', '.csv'),
            evaluation_range=(-ETA_CUT, ETA_CUT),
        )
        fb_systematic_csv = write_systematic_csv(
            fb_systematic,
            OUTPUT_DIR / output_name(f'{systematic_output_tag}_fb_relative').replace('.pdf', '.csv'),
            evaluation_range=(0.0, ETA_CUT),
        )

        annotations = (
            trigger, f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
            f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
            'p_{T}^{Lead} > 50 GeV', 'p_{T}^{SubLead} > 40 GeV',
            DIJET_DELTA_PHI_SELECTION_LABEL,
        )
        fit_text_eta = format_fit_summary_lines(eta_summaries) if SHOW_FIT_RESULTS else None
        fit_text_fb = format_fit_summary_lines(fb_summaries) if SHOW_FIT_RESULTS else None
        canvases = {
            'eta_overlay': draw_overlay(
                eta_shapes, title='', x_title='#eta_{CM}^{dijet}',
                y_title='1/N dN/d#eta_{CM}^{dijet}', x_range=eta_x_range,
                annotations=annotations, grid=DRAW_GRID, style_indices=SHAPE_STYLE,
                style=PLOT_STYLE, output=OUTPUT_DIR / output_name('full_overlay'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_overlay'),
            'fb_overlay': draw_overlay(
                fb_ratios, title='', x_title='#eta_{CM}^{dijet}', y_title='Forward / Backward',
                x_range=fb_x_range, y_range=FB_RANGE, reference_y=1.0,
                annotations=annotations, grid=DRAW_GRID, style_indices=SHAPE_STYLE,
                style=PLOT_STYLE, output=OUTPUT_DIR / output_name('fb_overlay'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_overlay'),
            'eta_ratios': draw_overlay(
                eta_variations, title='', x_title='#eta_{CM}^{dijet}',
                y_title='Pileup filter / dz1p0', x_range=eta_x_range,
                y_range=FULL_RATIO_RANGE, reference_y=1.0, annotations=annotations,
                grid=DRAW_GRID, overlay_functions=eta_functions, overlay_text=fit_text_eta,
                style_indices=RATIO_STYLE, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name('full_ratio_to_default'), save_png=SAVE_PNG,
                canvas_name=f'{common_tag}_full_ratio'),
            'fb_ratios': draw_overlay(
                fb_variations, title='', x_title='#eta_{CM}^{dijet}',
                y_title='(Forward / Backward) filter / dz1p0', x_range=fb_x_range,
                y_range=FB_DOUBLE_RATIO_RANGE, reference_y=1.0, annotations=annotations,
                grid=DRAW_GRID, overlay_functions=fb_functions, overlay_text=fit_text_fb,
                style_indices=RATIO_STYLE, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name('fb_ratio_to_default'), save_png=SAVE_PNG,
                canvas_name=f'{common_tag}_fb_ratio'),
            'eta_systematic': draw_overlay(
                {'Pileup systematic': eta_percent}, title='', x_title='#eta_{CM}^{dijet}',
                y_title='Pileup Rel. Syst. Uncrt. (%)', x_range=eta_x_range,
                y_range=SYSTEMATIC_Y_RANGE, annotations=annotations, grid=DRAW_GRID,
                style=PLOT_STYLE, output=OUTPUT_DIR / output_name(f'{systematic_output_tag}_full_relative'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_systematic'),
            'fb_systematic': draw_overlay(
                {'Pileup systematic': fb_percent}, title='', x_title='#eta_{CM}^{dijet}',
                y_title='Pileup Rel. Syst. Uncrt. (%)', x_range=fb_x_range,
                y_range=SYSTEMATIC_Y_RANGE, annotations=annotations, grid=DRAW_GRID,
                style=PLOT_STYLE, output=OUTPUT_DIR / output_name(f'{systematic_output_tag}_fb_relative'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_systematic'),
        }
        trigger_results[ptave_range] = {
            'eta_shapes': eta_shapes, 'fb_ratios': fb_ratios,
            'eta_variations': eta_variations, 'fb_variations': fb_variations,
            'eta_fit_functions': eta_functions, 'fb_fit_functions': fb_functions,
            'eta_fit_summaries': eta_summaries, 'fb_fit_summaries': fb_summaries,
            'eta_systematic_raw': eta_systematic_raw, 'fb_systematic_raw': fb_systematic_raw,
            'eta_systematic': eta_systematic, 'fb_systematic': fb_systematic,
            'eta_systematic_csv': eta_systematic_csv,
            'fb_systematic_csv': fb_systematic_csv,
            'selected_keys': selected_keys, 'canvases': canvases,
        }
        print(trigger, ptave_range, 'full fits:', eta_summaries)
        print(trigger, ptave_range, 'F/B fits:', fb_summaries)

    for ptave_range, result in trigger_results.items():
        low, high = ptave_range
        ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
        eta_cut_tag = int(round(10.0 * ETA_CUT))
        sample_tag = trigger.lower()
        common_tag = f'{sample_tag}_pileupSystematics_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        output_name = lambda plot: (
            f'{sample_tag}_pileupSystematics_{plot}'
            f'_{OUTPUT_CONFIGURATION_TAG}'
            f'_etaCM_{eta_cut_tag}_ptave_{ptave_tag}.pdf'
        )
        annotations = (trigger, f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV')
        result['eta_ratio_systematic_band'] = draw_ratio_with_systematic_band(
            result['eta_variations'], result['eta_systematic'],
            fit_functions=result['eta_fit_functions'], x_range=eta_x_range,
            y_range=FULL_RATIO_RANGE,
            output=OUTPUT_DIR / output_name('full_ratio_with_systematic_band'),
            canvas_name=f'{common_tag}_full_ratio_with_systematic_band',
            x_title='#eta_{CM}^{dijet}', annotations=annotations,
        )
        result['fb_ratio_systematic_band'] = draw_ratio_with_systematic_band(
            result['fb_variations'], result['fb_systematic'],
            fit_functions=result['fb_fit_functions'], x_range=fb_x_range,
            y_range=FB_DOUBLE_RATIO_RANGE,
            output=OUTPUT_DIR / output_name('fb_ratio_with_systematic_band'),
            canvas_name=f'{common_tag}_fb_ratio_with_systematic_band',
            x_title='#eta_{CM}^{dijet}', annotations=annotations,
        )
    return trigger_results


## MinimumBias data


In [ ]:
mb_results = analyze_trigger('MinimumBias')
pileup_results['MinimumBias'] = mb_results


## Jet60 data


In [ ]:
jet60_results = analyze_trigger('Jet60')
pileup_results['Jet60'] = jet60_results


## Jet80 data


In [ ]:
jet80_results = analyze_trigger('Jet80')
pileup_results['Jet80'] = jet80_results


## Jet100 data


In [ ]:
jet100_results = analyze_trigger('Jet100')
pileup_results['Jet100'] = jet100_results


## Validation summary

Check that normalized full-shape integrals are unity, F/B inputs were constructed with the protected independent-error option, and Vtx1 is absent from the systematic calculation.


In [ ]:
for trigger, trigger_results in pileup_results.items():
    for ptave_range, result in trigger_results.items():
        integrals = {label: hist.Integral() for label, hist in result['eta_shapes'].items()}
        assert all(abs(value - 1.0) < 1e-9 for value in integrals.values()), integrals
        assert 'Gplus / dz1p0' in result['eta_variations']
        # PyROOT represents a null TDirectory pointer as a falsey C++ proxy,
        # which is not necessarily identical to Python's None.
        assert not result['eta_systematic'].GetDirectory()
        assert not result['fb_systematic'].GetDirectory()
        print(trigger, ptave_range, integrals, result['selected_keys'])
